# Analyses from the master table

Every section below reads **only** `MASTER_PATH` — no joins to the annotation, association,
correlation or genebass files. Each section declares its own parameter block
(`variant_class`, config file, `selected_categories`, `mac`, …) so sections stay independent.

Shared setup is in §0: config loading, the derived-column helper, and the variant filter builder.

Master table built by [`utils/create_master_table.ipynb`](../utils/create_master_table.ipynb).
Grain is one row per `(id, region)`; gene/trait columns (`phenotype`, `loftee_corr_dir`, …)
are already attached to every variant row.

## 0. Shared setup

In [ ]:
import yaml
import numpy as np
import polars as pl

from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

import sys
from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'utils' / 'variant_filtering.py').exists())
sys.path.insert(0, str(REPO_ROOT))
from utils.variant_filtering import env_override, fetch_hf_data

MASTER_PATH = env_override('MASTER_PATH', fetch_hf_data('genebass_annotated.parquet', REPO_ROOT))
FIG_DIR     = env_override('FIG_DIR', '../../paper_figures')
CONFIG_DIR  = str(REPO_ROOT / 'configs')

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT))
from utils.variant_filtering import (
    load_config, load_variant_class, scan_variants, appv_of, pick_annos, filter_covered, derived_schema,
    gene_trait_tool_correlations, env_override,
)

_THEME = theme_minimal() + theme(
    axis_text=element_text(size=11, lineheight=1.4),
    axis_title=element_text(size=12),
    legend_text=element_text(size=12),
    legend_title=element_text(size=12),
    plot_background=element_rect(fill='white', color='white'),
)

MASTER_COLS = pl.scan_parquet(MASTER_PATH).collect_schema().names()
DERIVED_COLS = derived_schema(MASTER_PATH)
print(f'master table: {len(MASTER_COLS)} cols -> {len(DERIVED_COLS)} after add_derived()')


## 1. Phenotype correlations

Per (gene, trait, annotation) Spearman correlation between the annotation score and the
genebass beta, direction-corrected. Reproduces `pheno_correlations.ipynb`.

In [ ]:
# --- parameters (env_override(NAME, default) -- set UKBBGYM_NAME to override without editing)
C1_variant_class       = env_override('VARIANT_CLASS', 'missense')
C1_config               = env_override('CONFIG_FILE', 'config_correlations.yaml')
C1_selected_categories = env_override('SELECTED_CATEGORIES',
                                       ['missense', 'genetic_diversity', 'gnomad', 'conservation'], 'list')
C1_mac                 = env_override('MAC', 20, int)
C1_only_snps           = env_override('ONLY_SNPS', True, bool)
C1_only_clinvar        = env_override('ONLY_CLINVAR', False, bool)
C1_exclude_clinvar     = env_override('EXCLUDE_CLINVAR', False, bool)
C1_min_variants        = env_override('MIN_VARIANTS', 100, int)      # gene-trait pairs below this are dropped from the figure

In [ ]:
c1_cfg, c1_all = load_config(CONFIG_DIR, C1_config)
c1_vc  = load_variant_class(CONFIG_DIR, C1_variant_class)
c1_lf  = scan_variants(MASTER_PATH, c1_vc, only_snps=C1_only_snps,
                       only_clinvar=C1_only_clinvar, exclude_clinvar=C1_exclude_clinvar)
c1_annos = pick_annos(c1_cfg, c1_all, C1_selected_categories, DERIVED_COLS)

# Coverage-filtered per (gene, trait, tool) correlations -- the same canonical computation
# used by clinvar_spearman_scatterplot.ipynb, so the two analyses agree exactly
# on which genes qualify under the same tool set/mac/min_variants (see
# gene_trait_tool_correlations in utils/variant_filtering.py).
c1_filt = gene_trait_tool_correlations(c1_lf, C1_mac, c1_annos, c1_cfg, C1_selected_categories, C1_min_variants)
print(c1_filt.shape, f"{c1_filt['region'].n_unique()} genes")
c1_filt.head()

In [ ]:
# c1_filt already has the coverage + min-variants filter applied (see previous cell)
c1_filt = c1_filt.drop_nans().drop_nulls(subset=['corr_beta'])

# ---------------------------------------------------------------------------
# Pairwise head-to-head heatmap (see pheno_correlations.ipynb for the rationale).
# A tool "beats" another only if the paired win is positive AND significant.
# The axis order is a topological sort of that "significantly beats" relation, so
# no lower-ranked tool significantly beats a higher-ranked one. Ties among the
# currently-unbeaten tools break on: fewest losses, then most wins, then largest
# cumulative margin.
# ---------------------------------------------------------------------------
import itertools
import heapq
import pandas as pd
from scipy import stats

C1_ALPHA = 0.05

c1_tools = c1_filt['annotation'].unique().to_list()
c1_to_label = dict(c1_cfg.select(['annotation', 'label']).unique().iter_rows())

# 1. Every head-to-head paired comparison, computed once.
#    diff[(a, b)] = mean(corr_beta_a - corr_beta_b) over gene-trait pairs shared by a and b
#    pval[{a, b}] = Wilcoxon signed-rank p-value (symmetric)
c1_vals = {a: c1_filt.filter(pl.col('annotation') == a)
                     .select(['region', 'phenotype', 'corr_beta'])
           for a in c1_tools}
c1_diff, c1_pval = {}, {}
for a, b in itertools.combinations(c1_tools, 2):
    paired = c1_vals[a].join(c1_vals[b], on=['region', 'phenotype'], suffix='_b')
    ca, cb = paired['corr_beta'].to_numpy(), paired['corr_beta_b'].to_numpy()
    d = float(ca.mean() - cb.mean()) if len(ca) else 0.0
    if len(ca) >= 10 and not np.allclose(ca, cb):
        _, pv = stats.wilcoxon(ca, cb, alternative='two-sided')
    else:
        pv = 1.0
    c1_diff[(a, b)], c1_diff[(b, a)] = d, -d
    c1_pval[frozenset((a, b))] = pv

def c1_beats(a, b):   # a is significantly better than b on their shared gene-trait pairs
    return c1_pval[frozenset((a, b))] < C1_ALPHA and c1_diff[(a, b)] > 0

# 2. Upset-safe topological sort on the "beats" relation.
c1_wins   = {t: sum(c1_beats(t, u) for u in c1_tools if u != t) for t in c1_tools}
c1_losses = {t: sum(c1_beats(u, t) for u in c1_tools if u != t) for t in c1_tools}
c1_adv    = {t: sum(c1_diff[(t, u)] for u in c1_tools if u != t) for t in c1_tools}

def _c1_key(t):
    return (c1_losses[t], -c1_wins[t], -c1_adv[t], t)   # trailing `t` = deterministic tiebreak

c1_succ  = {t: [u for u in c1_tools if u != t and c1_beats(t, u)] for t in c1_tools}
c1_indeg = dict(c1_losses)          # in-degree in the full graph == total losses

c1_heap = [(_c1_key(t), t) for t in c1_tools if c1_indeg[t] == 0]
heapq.heapify(c1_heap)

c1_ordered, c1_placed = [], set()
while len(c1_ordered) < len(c1_tools):
    if c1_heap:
        _, t = heapq.heappop(c1_heap)
        if t in c1_placed:
            continue
    else:                                              # residual cycle -> force-break
        t = min((x for x in c1_tools if x not in c1_placed), key=_c1_key)
    c1_ordered.append(t)
    c1_placed.add(t)
    for u in c1_succ[t]:
        if u in c1_placed:
            continue
        c1_indeg[u] -= 1
        if c1_indeg[u] == 0:
            heapq.heappush(c1_heap, (_c1_key(u), u))

c1_labels = [c1_to_label.get(t, t) for t in c1_ordered]

# 3. Heatmap frame, reusing the precomputed comparisons.
c1_heat = pd.DataFrame([
    {'Tool_X': c1_to_label.get(x, x),
     'Tool_Y': c1_to_label.get(y, y),
     'mean_diff': 0.0 if x == y else c1_diff[(y, x)],           # mean(Tool_Y) - mean(Tool_X)
     'sig': '' if x == y else
            ('***' if c1_pval[frozenset((x, y))] < 0.001 else
             '**'  if c1_pval[frozenset((x, y))] < 0.01  else
             '*'   if c1_pval[frozenset((x, y))] < 0.05  else '')}
    for x, y in itertools.product(c1_ordered, repeat=2)
])
# Lock in categorical order so the best tools sit top/right
c1_heat['Tool_X'] = pd.Categorical(c1_heat['Tool_X'], categories=c1_labels,       ordered=True)
c1_heat['Tool_Y'] = pd.Categorical(c1_heat['Tool_Y'], categories=c1_labels[::-1], ordered=True)

c1_n = len(c1_ordered)
c1_plot = (
    ggplot(c1_heat, aes(x='Tool_X', y='Tool_Y', fill='mean_diff'))
    + geom_tile(color='#FFFFFF', size=0.5)
    + geom_text(aes(label='sig'), color='black', size=12, va='center', nudge_y=-0.1)
    + scale_fill_gradient2(low='#2C7BB6', mid='#FFFFFF', high='#D7191C', midpoint=0)
    + labs(subtitle=f"{c1_vc['x_label']} — {c1_filt.height // c1_n} gene-trait pairs",
           x='Tool X', y='Tool Y', fill='Tool Y − X\n(avg Spearman ρ)')
    + _THEME 
    + theme(
        figure_size=(c1_n * 0.4 + 1.5, c1_n * 0.4 + 2.5),
        aspect_ratio=1,
        axis_text=element_text(size=13),
        axis_text_x=element_text(rotation=45, hjust=1),
        axis_title=element_text(size=13),
        panel_grid=element_blank(),
        legend_position='bottom',
    )
)

if FIG_DIR:
    c1_plot.save(f'{FIG_DIR}/F4_master_pheno_corr_{C1_variant_class}_{C1_mac}.svg', dpi=200, verbose=False)

c1_plot